# Oturum 6 — `bentopy` ile Kalabalık Hücresel Sistemler

**Biyofizik 2026 Kursu · Dr. Öğr. Üyesi Ekrem Yaşar**

Bu not defterinde Martini resmî öğretim materyali doğrudan uygulanmaktadır:
[Bentopy Tutorial — cgmartini.nl](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)

Kurs boyunca izlenen ölçek gelişimi:

| Oturum | Sistem | Karakteristik uzunluk |
|---|---|---|
| 2 | Çözelti içinde tek lizozim molekülü | yaklaşık 5 nm |
| 3 | Membranda tek reseptör, atomistik | yaklaşık 10 nm |
| 4 | Membranda tek reseptör, kaba-taneli | yaklaşık 12 nm |
| 6 | Kalabalık, çok bölmeli hücresel sistem | 40 nm |

Uygulamada kullanılan model proteinlerden biri lizozimdir; Oturum 2'de
atomistik olarak hazırlanan sistemin kaba-taneli karşılığı burada yüzlerce
kopya hâlinde kullanılmaktadır.


---
## Kalabalık ortam koşullarının önemi

Moleküler simülasyonların büyük bölümü proteinleri seyreltik çözelti
koşullarında incelemektedir. Hücre içi ortam bu varsayımdan belirgin biçimde
ayrılmaktadır:

- Sitoplazmada toplam makromolekül derişimi yaklaşık 300 g/L düzeyindedir;
  hacmin %20–30'u makromoleküller tarafından işgal edilmektedir.
- Biyolojik membranlarda protein/lipit oranı yüksektir.
- Kalabalık koşulları difüzyonu yavaşlatmakta ve bağlanma dengelerini
  kaydırmaktadır.

**Yöntemsel güçlük.** Çok sayıda makromolekülün çakışma oluşturmadan, uygun
yönelimlerle, hedeflenen derişimde ve belirli hücresel bölmelere
yerleştirilmesi elle yapılabilecek bir işlem değildir.

### `bentopy` bileşenleri

| Komut | İşlevi |
|---|---|
| `bentopy-mask` | Var olan bir yapıdan bölme maskeleri üretilmesi |
| `bentopy-pack` | Yapıların bölmelere çakışmasız olarak yerleştirilmesi |
| `bentopy-render` | Yerleşim planından koordinat ve topolojinin üretilmesi |
| `bentopy-merge` | Paketlenen yapıların var olan bir sistemle birleştirilmesi |
| `bentopy-solvate` | Kalan boşluğun çözücü ve iyonlarla doldurulması |

Yerleşim, `.bent` uzantılı bir yapılandırma dosyasıyla tanımlanmaktadır.
Söz dizimi başvurusu:
[Reference for `.bent` files](https://github.com/marrink-lab/bentopy/wiki/Reference-for-bent)


---
## 1. Yazılım kurulumu


In [ ]:
%%capture
!pip install -q bentopy
!apt-get -qq update && apt-get -qq install -y gromacs


In [ ]:
!bentopy-pack --help 2>&1 | head -15
print('---')
!which bentopy-pack bentopy-render bentopy-solvate bentopy-mask bentopy-merge


---
## 2. Tutorial dosyalarının indirilmesi

Arşiv, uygulamalarda kullanılacak yapıları, topolojileri ve `.mdp`
dosyalarını içermektedir (yaklaşık 3.6 MB).


In [ ]:
!wget -q https://cgmartini-library.s3.ca-central-1.amazonaws.com/0_Tutorials/m3_tutorials/Bentopy/tutorial_files.tar.gz
!tar -xzf tutorial_files.tar.gz

import os
os.chdir('tutorial_files')
print('Calisma dizini:', os.getcwd())
!ls structures/ topology/ mdp_files/


---
## 3. Uygulama 1 — Kutu içinde protein paketleme

Sitoplazmik yoğunlukta, homojen dağılımlı bir protein sistemi kurulmaktadır:
40 × 40 × 40 nm boyutlarında bir kutuya 650 lizozim molekülü.

### Yapılandırma dosyası

`.bent` dosyası beş bölümden oluşmaktadır:

| Bölüm | İçeriği |
|---|---|
| `[ general ]` | Sistem başlığı ve rastgelelik tohumu |
| `[ space ]` | Kutu boyutları ve paketleme ızgara çözünürlüğü |
| `[ includes ]` | Topolojiye eklenecek kuvvet alanı dosyaları |
| `[ compartments ]` | Yerleştirmenin yapılacağı hacimlerin tanımı |
| `[ segments ]` | Hangi yapıdan kaç kopyanın hangi bölmeye konacağı |


In [ ]:
bent = '''[ general ]
title "Proteins in a box"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/lysozyme.itp"

[ compartments ]
system is all

[ segments ]
LYZ 650 from "structures/lysozyme.pdb" in system
'''
open('simple_packing.bent','w').write(bent)
print(bent)


### Paketleme, oluşturma ve solvatasyon


In [ ]:
!bentopy-pack simple_packing.bent placements.json


In [ ]:
!bentopy-render placements.json system.gro -t topol.top

import os
if os.path.exists('system.gro'):
    n = int(open('system.gro').read().splitlines()[1])
    print(f'Paketlenmis sistem: {n:,} parcacik')


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro \
    -s NA:0.15M -s CL:0.15M \
    --charge neutral \
    -t topol.top


In [ ]:
import os
if os.path.exists('solvated_system.gro'):
    n = int(open('solvated_system.gro').read().splitlines()[1])
    print(f'Solvatlanmis sistem: {n:,} parcacik')
    print()
    print('--- topol.top ---')
    print(open('topol.top').read())


**Tartışma sorusu.** Oturum 2'de tek bir lizozim molekülü için kurulan
atomistik sistemin atom sayısı ile bu sistemin parçacık sayısı
karşılaştırıldığında hangi ölçek farkı ortaya çıkmaktadır?


---
## 4. Uygulama 2 — Membran çevresinde konuma bağlı paketleme

Bu uygulamada proteinler yalnızca sayıca değil, konumsal kurala göre de
yerleştirilmektedir: lizozim çözücü hacmine dağıtılırken, ubikitin yalnızca
membran yüzeyine yakın bölgede konumlandırılmaktadır.

### Maskenin üretilmesi

Var olan bir membran yapısından bölme maskesi çıkarılmaktadır. İlk komut
etiketlemenin görsel olarak denetlenmesini sağlamakta, ikinci komut
kullanılacak maske dosyasını üretmektedir.


In [ ]:
!bentopy-mask structures/membrane.gro --visualize-labels labels.gro
!bentopy-mask structures/membrane.gro -l 1:membrane_mask.npz
!ls -lh membrane_mask.npz labels.gro


### Yapılandırma dosyası

**Kavramsal not.** `[ compartments ]` bölümündeki tanımlar:

- `membrane from "membrane_mask.npz"` — maske dosyasından bölme tanımlama
- `solvent combines not membrane` — membran dışında kalan hacim
- `close-to-membrane around 5 of membrane` — membran yüzeyinden 5 nm
  mesafedeki kabuk

Bu yaklaşım, periferik membran proteinlerinin fizyolojik dağılımının
modellenmesine olanak vermektedir.


In [ ]:
bent = '''[ general ]
title "Proteins around a membrane"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/martini_v3.0.0_phospholipids_v1.itp"
"topology/lysozyme.itp"
"topology/ubiquitin.itp"

[ compartments ]
membrane from "membrane_mask.npz"
solvent combines not membrane
close-to-membrane around 5 of membrane

[ segments ]
LYZ:lyz 300 from "structures/lysozyme.pdb" in solvent
UBQ:ubq 100 from "structures/ubiquitin.pdb" in close-to-membrane
'''
open('membrane_packing.bent','w').write(bent)
print(bent)


In [ ]:
!bentopy-pack membrane_packing.bent placements.json
!bentopy-render placements.json packed_proteins.gro -t topol.top


### Membranla birleştirme

**Dikkat.** `bentopy-merge` işleminden sonra lipit sayısının topoloji
dosyasına elle eklenmesi gerekmektedir; birleştirilen membran yapısı
`bentopy` tarafından üretilmediğinden topolojide otomatik olarak yer
almamaktadır.


In [ ]:
!bentopy-merge packed_proteins.gro structures/membrane.gro -o system.gro
!echo "POPC    5408" >> topol.top
!tail -8 topol.top


In [ ]:
!bentopy-solvate -i system.gro -o solvated_system.gro -t topol.top \
    -s NA:0.15M -s CL:0.15M --charge neutral

import os
if os.path.exists('solvated_system.gro'):
    n = int(open('solvated_system.gro').read().splitlines()[1])
    print(f'\nSolvatlanmis sistem: {n:,} parcacik')


---
## 5. Uygulama 3 — Çok bölmeli sistem

Çift membranla ayrılmış iki bölme tanımlanmakta ve her bölmeye farklı protein
yerleştirilmektedir. Süre elverdiği takdirde yürütülecektir.

Maskelerin üretilmesinde `-b` seçeneği bölme etiketlerinin görselleştirilmesini,
`-l` seçenekleri ise her etiket için ayrı maske dosyası üretilmesini
sağlamaktadır.


In [ ]:
!bentopy-mask structures/double_membrane.gro -b compartment_labels.gro
!bentopy-mask structures/double_membrane.gro \
    -l  -1:A_mask.npz \
    -l  -2:B_mask.npz \
    -l 1,2:membrane_mask.npz
!ls -lh A_mask.npz B_mask.npz membrane_mask.npz


In [ ]:
bent = '''[ general ]
title "Proteins in different compartments"
seed 0

[ space ]
dimensions 40, 40, 40
resolution 0.5

[ includes ]
"topology/martini_v3.0.0.itp"
"topology/martini_v3.0.0_ions_v1.itp"
"topology/martini_v3.0.0_solvents_v1.itp"
"topology/martini_v3.0.0_phospholipids_v1.itp"
"topology/lysozyme.itp"
"topology/ubiquitin.itp"

[ compartments ]
membrane from "membrane_mask.npz"
A from "A_mask.npz"
B from "B_mask.npz"
membrane-neighborhood around 4 of membrane
B-close-to-membrane combines membrane-neighborhood and B

[ segments ]
LYZ:lyz 200 from "structures/lysozyme.pdb" in A
UBQ:ubq 100 from "structures/ubiquitin.pdb" in B-close-to-membrane
'''
open('compartment_packing.bent','w').write(bent)
print(bent)


In [ ]:
!bentopy-pack compartment_packing.bent placements.json
!bentopy-render placements.json packed_proteins.gro -t topol.top
!bentopy-merge packed_proteins.gro structures/double_membrane.gro -o system.gro
!echo "POPC    10816" >> topol.top
!bentopy-solvate -i system.gro -o solvated_system.gro -t topol.top \
    -s NA:0.15M -s CL:0.15M --charge neutral


In [ ]:
import os
if os.path.exists('solvated_system.gro'):
    n = int(open('solvated_system.gro').read().splitlines()[1])
    print(f'Cok bolmeli sistem: {n:,} parcacik')


---
## 6. Kurulan sistemin simülasyona hazırlanması

Kursta üretim simülasyonu koşulmamaktadır. Aşağıdaki adımlar, kurulan
sistemin doğrudan kullanılabilir olduğunu göstermek amacıyla verilmiştir;
hücreler yorum satırı hâlinde bırakılmıştır.

Tam betik: [`06_bentopy/kodlar/simulasyon.sh`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/06_bentopy/kodlar/simulasyon.sh)


In [ ]:
# Enerji minimizasyonu
# !gmx grompp -f mdp_files/em.mdp -c solvated_system.gro -p topol.top -o em.tpr
# !gmx mdrun -v -deffnm em

# Indeks gruplarinin tanimlanmasi, dengeleme ve uretim asamalari icin
# kodlar/simulasyon.sh dosyasina bakiniz.


---
## Sorun giderme

| Sorun | Çözümü |
|---|---|
| `pip install bentopy` derleme hatası veriyor | Önceden derlenmiş paket bulunamamıştır; [rustup](https://rustup.rs/) ile Rust derleyicisi kurulmalıdır |
| `bentopy-pack: command not found` | Kurulum hücresi yeniden çalıştırılmalıdır |
| Maske dosyası üretilmiyor | Çalışma dizininin `tutorial_files/` olduğu doğrulanmalıdır |
| Paketleme çok uzun sürüyor | `[ segments ]` bölümündeki kopya sayısı azaltılabilir |

Oturum için ayrılan süre sınırlıdır; sorun yaşanması hâlinde uygulama
sonlandırılarak aşağıdaki materyaller kullanılacaktır:

- Komutlar ve `.bent` yapılandırma dosyaları: [`06_bentopy/kodlar/`](https://github.com/eygpcr/biyofizik2026-martini/tree/main/06_bentopy/kodlar)
- Uygulamanın tam kaydı: [`VIDEO.md`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/06_bentopy/VIDEO.md)

---

## Kaynaklar

- [Bentopy Tutorial — cgmartini.nl](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)
- [bentopy deposu](https://github.com/marrink-lab/bentopy) ve [wiki](https://github.com/marrink-lab/bentopy/wiki)
- [`.bent` dosya biçimi başvurusu](https://github.com/marrink-lab/bentopy/wiki/Reference-for-bent)
- [`bentopy-solvate` belgelendirmesi](https://github.com/marrink-lab/bentopy/blob/main/src/solvate/README.md)

Ayrıca bkz. [`ILERI_OKUMA.md`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/ILERI_OKUMA.md)
